In [16]:
!pip install -q transformers datasets accelerate scikit-learn

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
import pandas as pd

# Load datasets
train_df = pd.read_csv('/kaggle/input/hatespeechonwomen/trainV2.csv')
test_df = pd.read_csv('/kaggle/input/hatespeechonwomen/TestV2 - testV2.csv')

# Standardize labels: Non-Abusive -> 1, Abusive/abusive -> 0
label_map = {'Non-Abusive': 1, 'Abusive': 0, 'abusive': 0}
train_df['Class'] = train_df['Class'].map(label_map)

print("Label distribution:")
print(train_df['Class'].value_counts())

Label distribution:
Class
1    1883
0    1769
Name: count, dtype: int64


In [19]:
from huggingface_hub import login
from transformers import AutoTokenizer

login()

model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

In [20]:
from datasets import Dataset
import numpy as np

# Convert to HF Dataset
train_dataset = Dataset.from_pandas(train_df[['Text', 'Class']])
test_dataset = Dataset.from_pandas(test_df[['Text']])

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["Text"], padding="max_length", truncation=True, max_length=128)

# Map and Format
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("Class", "labels")
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask"])

# Split into Train/Val
final_dataset = tokenized_train.train_test_split(test_size=0.1, seed=42)
train_ds = final_dataset["train"]
val_ds = final_dataset["test"]

print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}")

Map:   0%|          | 0/3652 [00:00<?, ? examples/s]

Map:   0%|          | 0/913 [00:00<?, ? examples/s]

Train size: 3286, Val size: 366


In [21]:
import evaluate
from transformers import AutoModelForSequenceClassification

# Load model for 2 classes
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Setup F1 Metric
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/indic-bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",        # Changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=True, 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_55/3417083540.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [26]:
# Start the training process
trainer.train()

# After training, it's good practice to save the model locally
trainer.save_model("./best_indicbert_model")
print("Training complete and model saved!")

Epoch,Training Loss,Validation Loss,F1
1,No log,0.691333,0.663866
2,No log,0.610999,0.742169
3,0.662900,0.604700,0.766265


✅ Training complete and model saved!


In [29]:
# 1. Get predictions for the test dataset
predictions = trainer.predict(tokenized_test)

# 2. Extract the highest probability class for each row
import numpy as np
preds = np.argmax(predictions.predictions, axis=-1)

# 3. Create the submission DataFrame
# We map 1 back to 'Non-Abusive' and 0 back to 'Abusive'
output_df = pd.DataFrame({
    'Text': test_df['Text'],
    'Class': preds
})

# Inverse mapping to get the strings back
inverse_label_map = {1: 'Non-Abusive', 0: 'Abusive'}
output_df['Class'] = output_df['Class'].map(inverse_label_map)

# 4. Save to CSV
output_df.to_csv('predictions.csv', index=False)

print("Final predictions saved to 'predictions.csv'!")
output_df.head()

Final predictions saved to 'predictions.csv'!


,Text,Class
0,லக்ஷ்மி அம்மா நீங்க புலம்புங்க அவளுக அவளுகபாட்...,Abusive
1,"இன்னும் கைது பண்ணல... அனைத்து பெற்றோர்களும், க...",Abusive
2,"அப்பா,அம்மா, அந்த இன்டர்வியூ பண்ற வக்கிரம்புடி...",Abusive
3,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க கார...,Abusive
4,எல்லாமே script thaan. ஷகீலா உங்க scriptum அரும...,Non-Abusive
